<a href="https://colab.research.google.com/github/thegurdian/ML/blob/main/diabetes_project_actual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE  # Run: pip install imbalanced-learn

# 2. Load the dataset
df = pd.read_csv('/content/DiaBD_A Diabetes Dataset for Enhanced Risk Analysis and Research in Bangladesh.csv')

# ---------------------------------------------------------
# STEP 1: DATA CLEANING (Handling the Outliers & Zeros)
# ---------------------------------------------------------
# Fix impossible Glucose values (0) by replacing them with the median
glucose_median = df[df['glucose'] > 0]['glucose'].median()
df.loc[df['glucose'] == 0, 'glucose'] = glucose_median

# Fix extreme BMI outliers (like the 574 value) by replacing with the median
bmi_median = df[df['bmi'] < 100]['bmi'].median()
df.loc[df['bmi'] > 100, 'bmi'] = bmi_median

# ---------------------------------------------------------
# STEP 2: DATA PREPROCESSING
# ---------------------------------------------------------
# Convert text columns (Gender, Diabetic) into numbers (0 and 1)
df['gender'] = LabelEncoder().fit_transform(df['gender'])
df['diabetic'] = LabelEncoder().fit_transform(df['diabetic'])

# Separate our Features (X) from the Target we want to predict (y)
X = df.drop('diabetic', axis=1)
y = df['diabetic']

# Split the data into Training (80%) and Testing (20%) sets.
# 'stratify=y' ensures the testing set has the exact same ratio of diabetics as the training set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# ---------------------------------------------------------
# STEP 3: HANDLING CLASS IMBALANCE
# ---------------------------------------------------------
# Since only 6% of the dataset is diabetic, we use SMOTE (Synthetic Minority Oversampling Technique).
# SMOTE generates synthetic "fake" diabetic patients based on the real ones so the model has enough examples to learn from.
# CRITICAL: We only apply this to the Training data, never the Testing data!
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# ---------------------------------------------------------
# STEP 4: FEATURE SCALING
# ---------------------------------------------------------
# Machine learning algorithms perform best when all numbers are on a similar scale.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# STEP 5: MODEL TRAINING
# ---------------------------------------------------------
# We use Random Forest because it is highly resistant to overfitting and great with medical data.
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train_balanced)

# ---------------------------------------------------------
# STEP 6: EVALUATION
# ---------------------------------------------------------
y_pred = model.predict(X_test_scaled)

print(f"Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Detailed Classification Report (0 = No Diabetes, 1 = Diabetes):")
print(classification_report(y_test, y_pred))

Overall Accuracy: 90.36%

Detailed Classification Report (0 = No Diabetes, 1 = Diabetes):
              precision    recall  f1-score   support

           0       0.95      0.94      0.95       990
           1       0.28      0.32      0.30        68

    accuracy                           0.90      1058
   macro avg       0.62      0.63      0.62      1058
weighted avg       0.91      0.90      0.91      1058

